In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


# Personal Finance Dashboard - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on personal finance transactions to identify spending patterns and data quality issues before visualization in Tableau.


## 1. Load and Inspect Raw Transaction Data

Load transactions from CSV and examine the dataset structure, columns, and data types.


In [ ]:
# Load raw transaction data
df = pd.read_csv('../data/raw/transactions.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nFirst 10 rows:")
df.head(10)


In [ ]:
# Display basic statistics
print("Basic Information:")
df.info()
print("\n" + "="*50)
print("Summary Statistics:")
df.describe()


## 2. Data Quality Assessment

Analyze missing values, duplicates, and data type inconsistencies.


In [ ]:
# Check for missing values
print("Missing Values per Column:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Check for negative amounts
print(f"Negative amounts: {(df['amount'] < 0).sum()}")

# Convert and validate data types
print("\n" + "="*50)
print("Data Type Validation:")
try:
    df['date'] = pd.to_datetime(df['date'])
    print("✓ Dates successfully converted")
except Exception as e:
    print(f"✗ Date conversion error: {e}")

try:
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
    print("✓ Amounts successfully converted to numeric")
except Exception as e:
    print(f"✗ Amount conversion error: {e}")


## 3. Transaction Amount Distribution Analysis

Analyze the distribution of transaction amounts to identify patterns and outliers.


In [ ]:
# Calculate statistics for transaction amounts
print("Transaction Amount Statistics:")
print(f"Mean: ${df['amount'].mean():.2f}")
print(f"Median: ${df['amount'].median():.2f}")
print(f"Std Dev: ${df['amount'].std():.2f}")
print(f"Min: ${df['amount'].min():.2f}")
print(f"Max: ${df['amount'].max():.2f}")
print(f"Q1 (25%): ${df['amount'].quantile(0.25):.2f}")
print(f"Q3 (75%): ${df['amount'].quantile(0.75):.2f}")

# Create visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['amount'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transaction Amounts')
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df['amount'], vert=True)
axes[1].set_ylabel('Amount ($)')
axes[1].set_title('Box Plot of Transaction Amounts')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identify outliers using IQR method
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['amount'] < Q1 - 1.5*IQR) | (df['amount'] > Q3 + 1.5*IQR)]
print(f"\nOutliers detected (IQR method): {len(outliers)}")
if len(outliers) > 0:
    print(outliers)


## 4. Spending by Category Over Time

Analyze spending patterns across categories and time periods.


In [ ]:
# Spending by category
category_spending = df.groupby('category')['amount'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)
print("Spending by Category:")
print(category_spending)

# Visualize category spending
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of total spending by category
category_spending['sum'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Total Amount ($)')
axes[0].set_title('Total Spending by Category')
axes[0].grid(True, alpha=0.3, axis='x')

# Pie chart of spending distribution
axes[1].pie(category_spending['sum'], labels=category_spending.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Spending Distribution by Category')

plt.tight_layout()
plt.show()

# Time series of daily spending
daily_spending = df.groupby('date')['amount'].sum().sort_index()
plt.figure(figsize=(14, 5))
plt.plot(daily_spending.index, daily_spending.values, marker='o', linewidth=2, markersize=5, color='steelblue')
plt.xlabel('Date')
plt.ylabel('Daily Spending ($)')
plt.title('Daily Spending Trend')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Identify Spending Patterns and Anomalies

Detect recurring transactions, seasonal patterns, and unusual spending spikes.


In [ ]:
# Identify recurring transactions
print("Recurring Transactions (same description, amount):")
recurring = df.groupby(['description', 'amount']).size().reset_index(name='count')
recurring = recurring[recurring['count'] > 1].sort_values('count', ascending=False)
print(recurring)

# Day of week analysis
df['day_of_week'] = df['date'].dt.day_name()
day_spending = df.groupby('day_of_week')['amount'].agg(['sum', 'mean', 'count'])
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_spending = day_spending.reindex(day_order)
print("\n" + "="*50)
print("Spending by Day of Week:")
print(day_spending)

# Visualize day of week patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

day_spending['sum'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Total Spending ($)')
axes[0].set_title('Total Spending by Day of Week')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

day_spending['mean'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Average Spending ($)')
axes[1].set_title('Average Transaction Amount by Day of Week')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Identify spending spikes
daily_mean = daily_spending.mean()
daily_std = daily_spending.std()
threshold = daily_mean + 2 * daily_std
spikes = daily_spending[daily_spending > threshold]
print(f"\nSpending Spikes (>{threshold:.2f}):")
print(spikes)


## 6. Prepare Data for Visualization

Generate summary tables and export cleaned data for Tableau.


In [ ]:
# Create directory for cleaned data if it doesn't exist
Path('../data/cleaned').mkdir(parents=True, exist_ok=True)

# Save cleaned transaction data
df.to_csv('../data/cleaned/transactions_clean.csv', index=False)
print("✓ Saved cleaned transactions to data/cleaned/transactions_clean.csv")

# Create and save monthly summary
df['year_month'] = df['date'].dt.to_period('M')
monthly_summary = df.groupby(['year_month', 'category'])['amount'].sum().reset_index()
monthly_summary.columns = ['Month', 'Category', 'Amount']
monthly_summary['Month'] = monthly_summary['Month'].astype(str)
monthly_summary.to_csv('../data/cleaned/monthly_summary.csv', index=False)
print("✓ Saved monthly summary to data/cleaned/monthly_summary.csv")

# Create and save daily summary
daily_summary = df.groupby('date')['amount'].sum().reset_index()
daily_summary.columns = ['Date', 'Daily_Total']
daily_summary.to_csv('../data/cleaned/daily_summary.csv', index=False)
print("✓ Saved daily summary to data/cleaned/daily_summary.csv")

# Create and save category summary
category_summary = df.groupby('category').agg({
    'amount': ['sum', 'mean', 'count', 'min', 'max']
}).reset_index()
category_summary.columns = ['Category', 'Total', 'Average', 'Count', 'Min', 'Max']
category_summary.to_csv('../data/cleaned/category_summary.csv', index=False)
print("✓ Saved category summary to data/cleaned/category_summary.csv")

print("\n" + "="*50)
print("Data preparation complete! Files ready for Tableau.")
print("="*50)
